# Chapter 3 · Repository memory and initialization

The tasks are cut into twelve-step sessions, and each session starts with an empty conversation. Four ways of working are compared: a single 24-step session, two sessions with no state, two sessions with a progress file the agent writes, and two sessions with a handoff report the program prepares.

**Before you start, turn on the GPU:** menu *Runtime* → *Change runtime type* → *T4 GPU* → *Save*.

The full explanation is in the PDF chapter. This notebook is the lab.

## 1. Load the model, the repository and the tools

In [ ]:
!pip install -q transformers==5.19.0  # the versions the book was checked with

In [ ]:
"""Chapter 3: repository memory and initialization."""
import importlib
import io
import json
import re
import sys
import time
import urllib.request
import zipfile
from pathlib import Path

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

GPU_MODEL = "Qwen/Qwen3-4B-Instruct-2507"  # with Colab's free T4 GPU
CPU_MODEL = "Qwen/Qwen3-1.7B"              # without a GPU: smaller, and a worse agent
ROOT = Path.cwd()                          # where data/ and shared/ are
REPOSITORY = ("https://github.com/HernanDiaz/ai-agents-real-projects/"
              "archive/refs/heads/main.zip")
TASKS = ["T01", "T02", "T03", "T04", "T05", "T10", "T11", "T13", "T14", "T15",
         "T17", "T18", "T19", "T20"]        # the single-session tasks
MAX_STEPS = 30             # model calls per task
MAX_TOKENS_STEP = 600      # new tokens per call
INPUT_LIMIT = 12_000       # tokens of the conversation: more does not fit on the T4
INTACT_RESULTS = 2         # the latest tool results are never trimmed
MAX_REJECTIONS = 3         # times the gate sends the work back
rt = None                  # the tools module, after prepare()



def load(name=None):
    """Loads the model and its tokenizer: on the GPU, in float16; otherwise, on CPU."""
    gpu = torch.cuda.is_available()
    if name is None:
        name = GPU_MODEL if gpu else CPU_MODEL
    tok = AutoTokenizer.from_pretrained(name)
    model = AutoModelForCausalLM.from_pretrained(
        name, dtype=torch.float16 if gpu else torch.float32)
    model.to("cuda" if gpu else "cpu").eval()
    print(f"Model {name} on {'cuda' if gpu else 'cpu'}")
    return tok, model


def prepare():
    """The North Cycles repository, its tasks and the agent's tools. If they are
    missing (in Colab), they are downloaded from the book's repository."""
    global rt
    if not (ROOT / "shared" / "repo_tools.py").exists():
        with urllib.request.urlopen(REPOSITORY) as response:
            archive = zipfile.ZipFile(io.BytesIO(response.read()))
        for name in archive.namelist():
            relative = name.split("/", 1)[1]
            if relative.startswith(("en/data/", "en/shared/")) and name[-1] != "/":
                destination = ROOT / relative[len("en/"):]
                destination.parent.mkdir(parents=True, exist_ok=True)
                destination.write_bytes(archive.read(name))
    sys.path.insert(0, str(ROOT / "shared"))
    rt = importlib.import_module("repo_tools")
    print(f"Repository in {rt.REPO}: {len(rt.files_in(rt.REPO))} files")

In [ ]:
tok, model = load()
prepare()

## 2. The model

In [ ]:
def template(tok, messages):
    """The conversation as text, with the tools, in the model's chat template."""
    return tok.apply_chat_template(messages, tools=rt.SCHEMAS, tokenize=False,
                                   add_generation_prompt=True)


def generate(tok, model, text):
    """One answer from the model, with greedy decoding. Returns the text and the
    input and output tokens."""
    inputs = tok(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=MAX_TOKENS_STEP,
                                do_sample=False, temperature=None, top_p=None,
                                top_k=None, prefill_chunk_size=1024,
                                pad_token_id=tok.eos_token_id)
    n_input = inputs["input_ids"].shape[1]
    new = output[0, n_input:].cpu()
    del inputs, output
    if torch.cuda.is_available():
        torch.cuda.empty_cache()  # each step has a different length: without this,
        #                           reserved memory grows until the GPU is full
    return tok.decode(new, skip_special_tokens=True).strip(), n_input, len(new)


def read_calls(answer):
    """The calls {name, arguments} that the model writes between <tool_call> and
    </tool_call>. A call that cannot be understood gets name None."""
    calls = []
    for block in re.findall(r"<tool_call>(.*?)(?:</tool_call>|$)", answer, re.S):
        try:
            data = json.loads(block.strip(), strict=False)
        except ValueError:
            data = None
        if isinstance(data, dict) and "name" in data:
            arguments = data.get("arguments", {})
            calls.append({"name": data["name"],
                          "arguments": arguments if isinstance(arguments, dict)
                          else {}})
        else:
            calls.append({"name": None, "arguments": {}})
    return calls


def trim_conversation(tok, messages):
    """If the conversation goes over INPUT_LIMIT tokens, replaces the oldest tool
    results with a short notice. Returns the text for the model and how many it
    has trimmed (None if even that does not make it fit)."""
    text = template(tok, messages)
    trimmed = 0
    candidates = [m for m in messages if m["role"] == "tool"
                  and not m.get("trimmed")][:-INTACT_RESULTS or None]
    while len(tok(text)["input_ids"]) > INPUT_LIMIT:
        if not candidates:
            return text, None
        message = candidates.pop(0)
        message["content"] = json.dumps(
            {"notice": "Old result trimmed to save memory; ask for it again if you "
                       "need it.",
             "start": message["content"][:150]}, ensure_ascii=False)
        message["trimmed"] = True
        trimmed += 1
        text = template(tok, messages)
    return text, trimmed

## 3. The rules of chapter 2

In [ ]:
SYSTEM = (
    "You are a coding agent working on the repository of North Cycles, a bike "
    "store in Portland, Oregon. It is a management program in Python with several "
    "modules (orders, returns, stock, invoices, notices…), its tests in the tests/ "
    "folder and some sample data. You will get a request from someone at the store, "
    "and you have to solve it by changing the code with the tools: list files, read "
    "line ranges, search, edit by exact replacement, run the tests and a few allowed "
    "commands. Work step by step, one tool at a time. When you are done, answer "
    'without calling any tool, starting with "DONE:" and explaining in one or two '
    "sentences what you changed. If you cannot finish, start with \"NOT DONE:\" and "
    "explain why.")


AGENTS_RULES = """# AGENTS.md · North Cycles

Repository rules. Each one is checked at the end of your work.

R1. Before editing a file, read it with read_file.
R2. Do not change anything in data/.
R3. Change docs/ only if the request asks to change the terms and conditions.
R4. Do not delete or change existing tests. Exception: if the request changes a
    business rule, update the tests of the old rule and say so in your answer.
R5. Add your tests to the module's test file (tests/test_shipping.py for shipping.py):
    replace the line `if __name__ == "__main__":` with a new class containing your
    test, followed by that same line. Import what you need inside the test.
R6. Do not create new modules outside tests/.
R7. "DONE" requires: changes to the program code, a new check in tests/ and
    run_tests green after your last change.
R8. Your final answer names every file you changed.
"""


ASKS_DOCS = re.compile(r"(updat\w*|and in the)\s[^.]{0,40}terms\s+and\s+conditions",
                       re.I)


MENTIONS_TESTS = r"run_tests|(?<![\w/])tests?\b(?![/_])"  # "test", not a file name


def diff_lines(diff):
    """The added and removed lines of each file of a unified diff:
    {path: {"+": [...], "-": [...]}}."""
    files, current = {}, None
    for line in diff.splitlines():
        if line.startswith("+++ b/"):
            current = files.setdefault(line[6:], {"+": [], "-": []})
        elif line.startswith("--- a/") or current is None:
            continue
        elif line[:1] in "+-" and line[1:].strip():
            current[line[0]].append(line[1:])
    return files


def check_rules(task, log, changes, answer):
    """The broken rules, as {rule: evidence}. It looks at what has really changed
    (the diff), the log of calls and the final answer."""
    broken = {}
    lines = diff_lines(changes["diff"])
    read = set()
    for r in log:
        path = r["arguments"].get("path", "")
        if r["tool"] == "read_file" and r["ok"]:
            read.add(path)
        elif r["tool"] == "edit" and r["ok"] and path not in read:
            broken.setdefault("R1", f"you edited {path} without reading it first")
    for path in changes["changed"]:
        if path.startswith("data/"):
            broken["R2"] = f"you changed {path}"
        if path.startswith("docs/") and not ASKS_DOCS.search(task["request"]):
            broken["R3"] = f"you changed {path} and the request does not ask for it"
        if path.startswith("tests/") and path in changes["new"]:
            broken["R5"] = f"you created {path}; tests go in the module's test file"
        if not path.startswith(("tests/", "docs/", "data/")) and path in changes["new"]:
            broken["R6"] = f"you created the module {path}"
    removed = [r for r, l in lines.items() if r.startswith("tests/") and l["-"]]
    if removed and not re.search(MENTIONS_TESTS, answer or "", re.I):
        broken["R4"] = (f"you changed existing tests in {', '.join(removed)} "
                        "and your answer does not say so")
    if answer and says_done(answer):
        program = [r for r in changes["changed"]
                   if not r.startswith(("tests/", "docs/", "data/"))]
        new_test = any(re.search(r"def test_|assert", l)
                       for r, l2 in lines.items() if r.startswith("tests/")
                       for l in l2["+"])
        edits = [i for i, r in enumerate(log)
                 if r["tool"] in ("edit", "create_file") and r["ok"]]
        after = log[edits[-1] + 1:] if edits else log
        runs = [r for r in after if r["tool"] == "run_tests"]
        missing = [text for met, text in (
            (program, "there are no changes to the program code"),
            (new_test, "there is no new check in tests/"),
            (runs and runs[-1]["result"].get("all_pass"),
             "there is no green test run after your last change"))
            if not met]
        if missing:
            broken["R7"] = "; ".join(missing)
        unnamed = [r for r in changes["changed"]
                   if Path(r).name not in answer]
        if unnamed:
            broken["R8"] = f"your answer does not name {', '.join(unnamed)}"
    return broken


def says_done(answer):
    """True if the final answer says the task is done. An empty answer says
    nothing."""
    if not answer.strip():
        return False
    if re.search(r"NOT\s+DONE", answer, re.I):
        return False
    if re.search(r"DONE", answer):
        return True
    return not re.search(r"\bI (couldn[’']t|could not|can[’']t|cannot|was not able)"
                         r"|\bunable\b|not possible|\bI need\b|\?", answer, re.I)


class ToolsWithRules:
    """The agent's tools with the process rules checked before every edit: if the
    edit breaks R1, R2, R3, R5 or R6, it is not made and the agent gets the rule
    back. The other calls go through as they are."""

    def __init__(self, tools, task):
        self.tools = tools
        self.log = tools.log
        self.task = task
        self.read = set()

    def broken_rule(self, name, path):
        """The rule that editing or creating "path" would break, or None."""
        exists = self.tools.workspace.path(path).exists()
        if path.startswith("data/"):
            return "R2: do not change anything in data/."
        if path.startswith("docs/") and not ASKS_DOCS.search(self.task["request"]):
            return "R3: the request does not ask to change the terms and conditions."
        if name == "create_file" and path.startswith("tests/"):
            return ("R5: do not create test files; add your test to the module's test "
                    "file, replacing the line if __name__ == \"__main__\":")
        if name == "create_file" and not exists and not path.startswith(".agent/"):
            return "R6: do not create new modules outside tests/."
        if name == "edit" and exists and path not in self.read:
            return f"R1: read {path} with read_file before editing it."
        return None

    def call(self, name, arguments):
        """Like Tools.call, but checking the rules before editing."""
        path = str((arguments or {}).get("path", ""))
        if name in ("edit", "create_file"):
            rule = self.broken_rule(name, path)
            if rule:
                result = {"error": "Edit rejected by AGENTS.md. " + rule}
                self.log.append({"tool": name, "arguments": arguments,
                                 "ok": False, "seconds": 0,
                                 "result": result})
                return result
        result = self.tools.call(name, arguments)
        if name == "read_file" and "error" not in result:
            self.read.add(path)
        return result

## 4. The progress file and the handoff

The handoff report can be tried without the model, on a copy with the solution of a task.

In [ ]:
PROGRESS = ".agent/PROGRESS.md"


PROGRESS_RULE = """R9. Keep .agent/PROGRESS.md with three sections: "Done",
    "Pending" and "How to check". Create it when you start and update it after
    every change: the session may be cut off at any moment and another session
    will continue from there.
"""


END_WARNING = (f"Two steps left in this session. Update {PROGRESS} with what is done, "
               "what is pending and how to check it, so the next session can continue.")


def handoff(workspace, last_message):
    """The handoff report for the next session: what has changed (the diff), how the
    tests stand and the last thing the agent wrote in the previous session. The
    program prepares it, without the model and without relying on agent notes."""
    changes = rt.differences(workspace.source, workspace.folder)
    tests = rt.Tools(workspace).run_tests()
    lines = ["Handoff report (prepared by the program at the start of the session):",
             "- Files changed from the original: "
             + (", ".join(changes["changed"]) or "none") + "."]
    if changes["diff"]:
        lines.append("- Changes made so far (diff):\n" + changes["diff"][:2000])
    lines.append(f"- Tests: {tests['result']} ({tests['ran']} run).")
    if not tests["all_pass"]:
        lines.append("  End of the test output:\n" + tests["output"][-800:])
    if last_message:
        lines.append("- What the previous session wrote last:\n" + last_message[:600])
    return "\n".join(lines)

In [ ]:
task = rt.read_task("T01")
with rt.Workspace() as workspace:
    rt.apply_solution(rt.Tools(workspace), task["folder"] / "solution")
    print(handoff(workspace, "I changed shipping_cost; it still needs checking."))

## 5. Several sessions

The task of the new tune-up price, with no state and with the handoff. Compare the first calls of the second session.

In [ ]:
def session(tok, model, task, workspace, tools, messages, max_steps, warn=False,
            verbose=False):
    """One session of the agent loop, with the rules gate of chapter 2, on a
    repository that may come already worked on. With "warn", two steps before the
    end it asks the agent to update the progress file. Returns the final answer
    (None if the steps run out), the steps and the tokens."""
    final, steps, n_in, n_out, rejections = None, 0, 0, 0, 0
    for steps in range(1, max_steps + 1):
        if warn and steps == max_steps - 1:
            messages.append({"role": "user", "content": END_WARNING})
        text, trimmed = trim_conversation(tok, messages)
        if trimmed is None:
            break
        answer, n_input, n_output = generate(tok, model, text)
        n_in, n_out = n_in + n_input, n_out + n_output
        calls = read_calls(answer)
        if not calls:
            messages.append({"role": "assistant", "content": answer})
            if verbose:
                print(f"[{steps}] {answer}")
            if says_done(answer) and rejections < MAX_REJECTIONS:
                changes = rt.differences(workspace.source, workspace.folder)
                broken = check_rules(task, tools.log, changes, answer)
                if broken:
                    rejections += 1
                    notice = ("It isn't done yet according to AGENTS.md: "
                              + "; ".join(f"{r}: {t}"
                                          for r, t in sorted(broken.items()))
                              + ". Keep working.")
                    messages.append({"role": "user", "content": notice})
                    continue
            final = answer
            break
        thought = answer.split("<tool_call>")[0].strip()
        messages.append({"role": "assistant", "content": thought,
                         "tool_calls": [{"type": "function", "function": c}
                                        for c in calls if c["name"]]})
        for call in calls:
            if call["name"] is None:
                result = {"error": "The tool call cannot be understood."}
            else:
                result = tools.call(call["name"], call["arguments"])
            if verbose:
                print(f"[{steps}] {call['name']}"
                      f"({json.dumps(call['arguments'], ensure_ascii=False)[:90]})"
                      f" -> {json.dumps(result, ensure_ascii=False)[:110]}")
            messages.append({"role": "tool", "name": call["name"] or "?",
                             "content": json.dumps(result, ensure_ascii=False)})
    return final, steps, n_in, n_out


def agent(tok, model, task, sessions=2, steps_per_session=12, progress=False,
          with_handoff=False, verbose=False):
    """One task in one or two sessions on the same repository. Every session starts
    with an empty conversation. With "progress", the AGENTS.md asks the agent to keep
    .agent/PROGRESS.md; with "with_handoff", the second session starts with the
    handoff report the program prepares. Returns the row of results."""
    start = time.time()
    instructions = AGENTS_RULES + (PROGRESS_RULE if progress else "")
    system = SYSTEM + "\n\n" + instructions
    session_rows, final, last = [], None, ""
    with rt.Workspace() as workspace:
        base = rt.Tools(workspace)
        for number in range(1, sessions + 1):
            request = task["request"]
            if number > 1:
                request += ("\n\nAnother session already started working on this "
                            "request and was cut off. Continue from where it left off.")
                if progress:
                    request += f" Read {PROGRESS} before you continue."
                if with_handoff:
                    request += "\n\n" + handoff(workspace, last)
            messages = [{"role": "system", "content": system},
                        {"role": "user", "content": request}]
            tools = ToolsWithRules(base, task)  # with no earlier reads
            before = len(base.log)
            if verbose:
                print(f"--- session {number}")
            final, steps, n_in, n_out = session(
                tok, model, task, workspace, tools, messages, steps_per_session,
                warn=progress and number < sessions, verbose=verbose)
            last = next((m["content"] for m in reversed(messages)
                         if m["role"] == "assistant" and m["content"]), "")
            session_rows.append({"steps": steps, "input_tokens": n_in,
                                 "output_tokens": n_out,
                                 "calls": base.log[before:]})
            if final is not None and says_done(final):
                break                           # done: no need for another session
        evaluation = rt.evaluate_attempt(workspace, task)
        changes = rt.differences(workspace.source, workspace.folder)
        broken = check_rules(task, base.log, changes, final)
        has_progress = workspace.path(PROGRESS).exists()
    done = final is not None and says_done(final)
    second = session_rows[1]["calls"] if len(session_rows) > 1 else []
    read_before = {r["arguments"].get("path") for r in session_rows[0]["calls"]
                   if r["tool"] == "read_file" and r["ok"]}
    return {"task": task["id"], "type": task["meta"]["type"],
            "solved": evaluation["solved"], "says_done": done,
            "false_done": done and not evaluation["solved"],
            "broken_rules": broken, "sessions": len(session_rows),
            "progress_written": has_progress,
            "rereads": sum(r["tool"] == "read_file"
                           and r["arguments"].get("path") in read_before
                           for r in second),
            "second_session_calls": len(second),
            "calls_before_edit": next(
                (i for i, r in enumerate(second, 1)
                 if r["tool"] in ("edit", "create_file") and r["ok"]
                 and not r["arguments"].get("path", "").startswith(".agent/")), None),
            "steps": sum(s["steps"] for s in session_rows),
            "input_tokens": sum(s["input_tokens"] for s in session_rows),
            "output_tokens": sum(s["output_tokens"] for s in session_rows),
            "seconds": round(time.time() - start, 1),
            "answer": final or "(no final answer)",
            "calls": [[[r["tool"], r["arguments"].get("path", ""), r["ok"]]
                       for r in s["calls"]] for s in session_rows],
            "diff": evaluation["diff"]}


VARIANTS = {
    "one session": {"sessions": 1, "steps_per_session": 24},
    "no state": {"sessions": 2, "steps_per_session": 12},
    "with progress file": {"sessions": 2, "steps_per_session": 12, "progress": True},
    "with handoff": {"sessions": 2, "steps_per_session": 12, "with_handoff": True},
}

In [ ]:
for name in ("no state", "with handoff"):
    print(f"=== {name}")
    row = agent(tok, model, rt.read_task("T17"), verbose=True, **VARIANTS[name])
    print("Solved:", row["solved"], "· Rereads:", row["rereads"])

## 6. The seventeen tasks with the four variants

By default a sample is run; with `COMPLETE = True`, the book's whole experiment.

In [ ]:
CH3_TASKS = TASKS + ["T09", "T12", "T16"]   # with the three two-session tasks


def summary(rows):
    """The totals of a variant."""
    two = [r for r in rows if r["sessions"] == 2]
    return {"tasks": len(rows),
            "solved": sum(r["solved"] for r in rows),
            "says_done": sum(r["says_done"] for r in rows),
            "false_done": sum(r["false_done"] for r in rows),
            "broken_rules": sum(len(r["broken_rules"]) for r in rows),
            "with_second_session": len(two),
            "solved_in_second": sum(r["solved"] for r in two),
            "rereads": sum(r["rereads"] for r in two),
            "progress_written": sum(r["progress_written"] for r in rows),
            "steps": sum(r["steps"] for r in rows),
            "input_tokens": sum(r["input_tokens"] for r in rows),
            "seconds": round(sum(r["seconds"] for r in rows), 1)}


def run_all(tok, model, options, tasks=CH3_TASKS):
    """One variant with every task: its rows and its summary."""
    rows = []
    for task_id in tasks:
        row = agent(tok, model, rt.read_task(task_id), **options)
        rows.append(row)
        print(f"  {task_id} {'solved' if row['solved'] else 'no    '}"
              f"  {'says done' if row['says_done'] else 'not said '}"
              f"  sessions={row['sessions']}  rereads={row['rereads']}"
              f"  steps={row['steps']:2d}  {row['seconds']:6.1f} s", flush=True)
    return {"rows": rows, "summary": summary(rows)}

In [ ]:
# With COMPLETE = False, the new-price task is run (about 30 minutes on Colab's T4);
# with True, the chapter's whole experiment (about twenty hours on the T4).
COMPLETE = False
tasks = CH3_TASKS if COMPLETE else ["T17"]
results = {}
for name, options in VARIANTS.items():
    print(f"== {name}")
    results[name] = run_all(tok, model, options, tasks=tasks)
    print(results[name]["summary"])

The book's full results, with all the tasks, are saved in `data/results/chapter_3/`, which `prepare` has downloaded:

In [ ]:
import json
saved = json.loads((ROOT / "data" / "results" / "chapter_3" / "results.json").read_text(encoding="utf-8"))
for name, r in saved.items():
    print(name, r["summary"])

## Exercises

The results files the exercises cite are in `data/results/chapter_3/`, which `prepare` downloads with the data.

1. List what a person who picks up a colleague's half-done task would need to know. Which of those things can a program get from the repository, and which are known only to whoever did the work? *(No GPU.)*

2. Run the sales tax task (T04) with the handoff and `verbose=True`. Copy the report the second session receives. What does it say about the state of the task, and what does the second session do with it? *(GPU, about 15 minutes.)*

3. Remove the diff from the handoff report and leave only the changed files, the tests and the last message. How many tasks does it solve? Which part of the report weighs the most? The results with the seventeen tasks are already in `results_exercises.json`; to see it for yourself, three are enough, for example T04, T10 and T15. *(No GPU with the saved results; GPU, about 50 minutes, with those tasks.)*

4. In the progress variant, the agent almost always ignored rule R9 and the warning. Write a check that does not let the first session finish without `.agent/PROGRESS.md`, like the process rules of chapter 2, and try it with the sales tax task (T04) and the new-price task (T17). Does it write it then? Does it help the second session? *(GPU, about 20 minutes.)*

5. Split the tasks into three eight-step sessions with the handoff. How many does it solve? Where is the limit below which the sessions are too short? The results with the seventeen tasks are already in `results_exercises.json`; to see it for yourself, three are enough, for example T04, T10 and T15. *(No GPU with the saved results; GPU, about 30 minutes, with those tasks.)*

6. The report cuts the diff at 2,000 characters. Find the task with the longest diff when the second session starts and look at what is lost with the cut. Propose another way of summarizing a long diff. *(No GPU.)*

7. The handoff includes the agent's last message. Find a case in which that message says something false about what the first session did. Did the second session notice when it saw the diff? The saved results keep the final answers, not the last message of each session, so you have to run a task with `verbose=True`. *(GPU, about 15 minutes per task.)*

8. The three two-part tasks (paying in installments, invoice numbering and holiday returns) are not solved with any variant. Read the trace of one of them with the handoff and say whether the second session got to the second part. *(GPU, between 10 and 30 minutes, depending on the task.)*

9. Compute the input tokens per solved task of the four variants. How much does the handoff report take up compared with the rest of the second session's conversation? *(No GPU.)*

10. **Project.** Write a startup script for a repository of yours that, when you start working, shows the uncommitted changes, the result of the tests and the latest notes from the git history. Use it for a week at the start of every session with a coding agent and note whether it changes how the session starts.

## References

Model [Qwen3-4B-Instruct-2507](https://huggingface.co/Qwen/Qwen3-4B-Instruct-2507). [Effective harnesses for long-running agents](https://www.anthropic.com/engineering/effective-harnesses-for-long-running-agents) (Anthropic, 2025). The North Cycles store, its program and its tasks are fictional. Original code and experiments.